<div style="background:linear-gradient(135deg, rgba(249,112,102,0.18), rgba(249,112,102,0.02)); border-left:6px solid #F97066; border-radius:10px; padding:20px 24px; margin-bottom:20px;">
<h1 style="margin:0; color:#F97066; font-size:1.8em;">🧠 Memoria y Estado en Agentes</h1>
<p style="margin:6px 0 0; opacity:0.8;">Unidad 5 — Por qué un agente olvida, y cómo darle memoria persistente entre turnos</p>
</div>

En <code>1-fundamentos-agentes.ipynb</code> cada llamada a <code>agente.invoke(...)</code> fue independiente: una pregunta, una respuesta. En una conversación real, el agente necesita recordar lo que se dijo antes. Este notebook muestra primero el problema (un agente que olvida entre invocaciones) y luego cómo resolverlo con el <strong>checkpointer</strong> de LangGraph.

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0 24px; background:rgba(14,165,233,0.04);">
<strong>📑 Contenido de esta guía</strong>
<ol style="margin:8px 0 0; padding-left:20px;">
<li><a href="#problema-sin-memoria">El problema: un agente sin memoria</a></li>
<li><a href="#checkpointer">Memoria con un checkpointer</a></li>
<li><a href="#aislamiento-hilos">Aislamiento entre conversaciones (thread_id)</a></li>
<li><a href="#cierre">Cierre y próximos pasos</a></li>
</ol>
</div>

<a id="problema-sin-memoria"></a>

## <span style="color:#F97066;">El problema: un agente sin memoria</span>

Un agente construido con <code>create_agent</code>, sin ninguna configuración adicional, no recuerda nada entre una llamada a <code>invoke</code> y la siguiente: cada invocación arranca con la lista de <code>messages</code> que se le pasa explícitamente, sin rastro de invocaciones anteriores.

In [1]:
import os
import logging
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

load_dotenv()

# Silenciar un aviso benigno de la librería sobre el uso de function calling automático
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    api_key=os.getenv("GOOGLE_API_KEY"),
)

agente_sin_memoria = create_agent(
    model=llm,
    tools=[],
    system_prompt="Eres un asistente breve.",
)

print("Agente sin memoria construido correctamente.")

Agente sin memoria construido correctamente.


In [2]:
# 1. Primer turno: el usuario se presenta
r1 = agente_sin_memoria.invoke({"messages": [{"role": "user", "content": "Hola, me llamo Sebastian."}]})
print("Turno 1:", r1["messages"][-1].text)

# 2. Segundo turno: una invocación nueva e independiente, sin el historial del turno 1
r2 = agente_sin_memoria.invoke({"messages": [{"role": "user", "content": "¿Cuál es mi nombre?"}]})
print("Turno 2:", r2["messages"][-1].text)

Turno 1: ¡Hola, Sebastian! ¿En qué puedo ayudarte hoy?


Turno 2: No lo sé. ¿Cómo te llamas?


El segundo turno no tiene forma de saber el nombre: la invocación no incluyó el mensaje del primer turno, y el agente no guardó nada por su cuenta. Para que un agente sostenga una conversación de varios turnos, algo tiene que persistir el historial de mensajes entre invocaciones — eso es exactamente lo que hace un <strong>checkpointer</strong>.

<a id="checkpointer"></a>

## <span style="color:#F97066;">Memoria con un checkpointer</span>

Un <strong>checkpointer</strong> de LangGraph guarda, después de cada paso del grafo, el estado completo de la conversación (la lista de <code>messages</code>) asociado a un identificador de conversación llamado <strong>thread_id</strong>. En la siguiente invocación con el mismo <code>thread_id</code>, LangGraph recupera ese estado y lo antepone a los mensajes nuevos, así el agente "ve" toda la conversación.

<code>InMemorySaver</code> guarda ese estado en la memoria RAM del proceso — perfecto para aprender y para pruebas, pero se pierde al reiniciar el proceso. En producción se usa un checkpointer persistente (por ejemplo <code>SqliteSaver</code> o un checkpointer respaldado por Postgres) que guarda el estado en disco o en una base de datos.

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

# 1. Un checkpointer en memoria: guarda el estado de cada conversación mientras el proceso viva
checkpointer = InMemorySaver()

# 2. El mismo agente de antes, ahora con memoria
agente_con_memoria = create_agent(
    model=llm,
    tools=[],
    system_prompt="Eres un asistente breve.",
    checkpointer=checkpointer,
)

print("Agente con memoria construido correctamente.")

Agente con memoria construido correctamente.


Cada invocación ahora debe indicar a qué conversación pertenece, a través de <code>config={"configurable": {"thread_id": "..."}}</code>.

In [4]:
config_conversacion_1 = {"configurable": {"thread_id": "conversacion-1"}}

# 1. Primer turno de la conversación "conversacion-1"
r1 = agente_con_memoria.invoke(
    {"messages": [{"role": "user", "content": "Hola, me llamo Sebastian."}]},
    config=config_conversacion_1,
)
print("Turno 1:", r1["messages"][-1].text)

# 2. Segundo turno de la MISMA conversación: el checkpointer recuperó el historial automáticamente
r2 = agente_con_memoria.invoke(
    {"messages": [{"role": "user", "content": "¿Cuál es mi nombre?"}]},
    config=config_conversacion_1,
)
print("Turno 2:", r2["messages"][-1].text)

Turno 1: ¡Hola, Sebastian! ¿En qué puedo ayudarte hoy?


Turno 2: Te llamas Sebastian.


Esta vez el agente sí recuerda: no porque el LLM "aprendiera" nada, sino porque el checkpointer reconstruyó el historial completo de la conversación antes de la segunda llamada.

<div style="border-left:4px solid #6366F1; background:rgba(99,102,241,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>📝 Nota</strong><br>
El historial completo se reenvía al LLM en cada turno — no hay "memoria" dentro del modelo entre llamadas a la API. Esto tiene una consecuencia práctica: conversaciones muy largas consumen cada vez más tokens por turno, ya que todo el historial viaja de nuevo en cada invocación.
</div>

<a id="aislamiento-hilos"></a>

## <span style="color:#F97066;">Aislamiento entre conversaciones (thread_id)</span>

El mismo agente con el mismo checkpointer puede sostener muchas conversaciones en paralelo, cada una identificada por su propio <code>thread_id</code>. Un <code>thread_id</code> distinto es, para el checkpointer, una conversación completamente distinta — sin acceso al historial de las demás.

In [5]:
config_conversacion_2 = {"configurable": {"thread_id": "conversacion-2"}}

# Misma pregunta, pero en un hilo (thread_id) que nunca mencionó ningún nombre
r3 = agente_con_memoria.invoke(
    {"messages": [{"role": "user", "content": "¿Cuál es mi nombre?"}]},
    config=config_conversacion_2,
)
print("Conversación 2, turno 1:", r3["messages"][-1].text)

Conversación 2, turno 1: No lo sé. ¿Cómo te llamas?


El agente no sabe el nombre en esta conversación, aunque sí lo sabía en <code>conversacion-1</code> — cada <code>thread_id</code> mantiene su propio estado, completamente aislado. Esto es lo que permite que una misma aplicación (por ejemplo, un chatbot de soporte) atienda a muchos usuarios distintos a la vez sin mezclar sus conversaciones.

---

<a id="cierre"></a>

# <span style="color:#F97066;">🎯 Cierre y próximos pasos</span>

<div style="border-left:4px solid #14B8A6; background:rgba(20,184,166,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>✅ Resumen</strong><br>
Este notebook mostró la memoria de los agentes:

- Sin un checkpointer, cada invocación de un agente es independiente — el agente olvida todo entre turnos.
- Un <code>checkpointer</code> (por ejemplo <code>InMemorySaver</code>) persiste el historial de mensajes entre invocaciones, asociado a un <code>thread_id</code>.
- Cada <code>thread_id</code> es una conversación aislada: el mismo agente puede sostener muchas conversaciones en paralelo sin mezclarlas.
- En producción se usa un checkpointer persistente (disco o base de datos) en vez de <code>InMemorySaver</code>, para no perder el historial al reiniciar el proceso.

</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0; background:rgba(14,165,233,0.04);">
<strong>➡️ Continúe con</strong>
<ul style="margin:8px 0 0; padding-left:20px;">
<li><code>3-multiples-herramientas.ipynb</code> — un agente con varias herramientas distintas, y cómo elige cuál usar en cada caso.</li>
</ul>
</div>